In [1]:
from pyscf import gto, dft, sftda
import sys
sys.path.append('/home/peter/Research/ZeroExcitationEnergy/code/TwoGGAs')
from GGAnew import eval_xc, td, verify

atom = '''
6  0.000000    0.000000    0.103187
1 -0.000000    0.995526   -0.309561
1 -0.000000   -0.995526   -0.309561
'''
mol = gto.M(atom=atom, charge=0, spin=2, basis='6-31G', verbose=9, symmetry=True)
mf = dft.UKS(mol)
mf = mf.define_xc_(eval_xc, 'GGA')
# mf.xc = 'B3LYP'
mf.grids.level = 1
mf.conv_tol = 1e-12
mf.conv_tol_grad = 1e-8
mf.kernel()

System: uname_result(system='Linux', node='peter7920', release='5.15.0-136-generic', version='#147~20.04.1-Ubuntu SMP Wed Mar 19 16:13:14 UTC 2025', machine='x86_64')  Threads 40
Python 3.9.21 (main, Dec 11 2024, 16:24:11) 
[GCC 11.2.0]
numpy 1.26.4  scipy 1.13.1  h5py 3.13.0
Date: Sat May  3 09:47:01 2025
PySCF version 2.8.0
PySCF path  /home/peter/anaconda3/envs/sftdgrad/lib/python3.9/site-packages/pyscf

[CONFIG] ARGPARSE = False
[CONFIG] DEBUG = False
[CONFIG] MAX_MEMORY = 4000
[CONFIG] TMPDIR = /tmp
[CONFIG] UNIT = angstrom
[CONFIG] VERBOSE = 3
[CONFIG] conf_file = None
[INPUT] verbose = 9
[INPUT] max_memory = 4000 
[INPUT] num. atoms = 3
[INPUT] num. electrons = 8
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 2
[INPUT] symmetry True subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 C      0.000000000000   0.000000000000   0.10318

-33.216406249149074

In [2]:
import numpy as np
import sys
sys.path.append('/home/peter/Research/ZeroExcitationEnergy/code/TwoGGAs')
from GGAnew import eval_xc, td, verify, eval_ab_sf, eval_fmat_g1


dm = mf.make_rdm1()
A_abab, A_baba, B_abba, B_baab = eval_ab_sf(mf, dm, eval_fmat_g1)

n_occ_a, n_occ_b = mf.mol.nelec
n_virt_a, n_virt_b = mf.mol.nao_nr() - n_occ_a, mf.mol.nao_nr() - n_occ_b

Cas = np.block([[A_baba, B_baab], [-B_abba.conj(), -A_abab.conj()]])
eigenvals, eigenvecs = np.linalg.eig(Cas)

idxt = eigenvals.real.argsort()
eigenvals = eigenvals[idxt]
eigenvecs = eigenvecs[:, idxt]

norms = np.linalg.norm(eigenvecs[:n_occ_a * n_virt_b], axis=0)**2
norms -= np.linalg.norm(eigenvecs[n_occ_a * n_virt_b:], axis=0)**2

sfd_eigenvals = eigenvals[norms>0]
sfd_eigenvecs = eigenvecs[:, norms>0].transpose(1,0)

def norm_xy(z):
    x = z[:n_occ_a*n_virt_b].reshape(n_occ_a,n_virt_b)
    y = z[n_occ_a*n_virt_b:].reshape(n_occ_b,n_virt_a)
    norm = np.linalg.norm(x)**2 - np.linalg.norm(y)**2
    norm = np.sqrt(1./norm)
    return ((0,x*norm), (y*norm,0))

mftd1 = sftda.TDA_SF(mf)
mftd1.nstates = len(sfd_eigenvals)
mftd1.extype = 1
mftd1.collinear_samples = 200
mftd1.e = sfd_eigenvals
mftd1.xy = [norm_xy(z) for z in sfd_eigenvecs]

In [3]:
mf.mo_energy

(array([-8.09682976, -0.20798961, -0.04902282,  0.12518487,  0.14684619,
         0.32966285,  0.38715637,  0.78903307,  0.80445428,  0.83685957,
         0.90821709,  1.24486856,  1.30119876]),
 array([-8.18975542, -0.2120569 , -0.04887048,  0.12249275,  0.14515545,
         0.32785177,  0.38726029,  0.78486812,  0.79958153,  0.83719719,
         0.90557873,  1.24249662,  1.30149291]))

In [4]:
from pyscf.sftda.tools_td import transition_analyze

print(transition_analyze(mf, mftd1, mftd1.e[0], mftd1.xy[0], tdtype='TDDFT'))
print(transition_analyze(mf, mftd1, mftd1.e[1], mftd1.xy[1], tdtype='TDDFT'))
print(transition_analyze(mf, mftd1, mftd1.e[2], mftd1.xy[2], tdtype='TDDFT'))
print(transition_analyze(mf, mftd1, mftd1.e[3], mftd1.xy[3], tdtype='TDDFT'))
print(transition_analyze(mf, mftd1, mftd1.e[4], mftd1.xy[4], tdtype='TDDFT'))

Excited energy : -0.6494078238961879 eV.
<S2> value : 1.999981247719423.
The main and second norm:
0.9998732867456795 0.00012363563656538269
The main and second norm to orbital pair:
(1.0, 5.0) (1.0, 9.0)


None
Excited energy : -0.026780198144807185 eV.
<S2> value : 2.0000089090042845.
The main and second norm:
0.4999981436334851 0.4999671296898288
The main and second norm to orbital pair:
(1.0, 10.0) (1.0, 4.0)


None
Excited energy : 3.760273938101146e-09 eV.
<S2> value : 2.0000010564955755.
The main and second norm:
0.4999902126251534 0.49994417361334087
The main and second norm to orbital pair:
(1.0, 10.0) (1.0, 4.0)


None
Excited energy : 0.5569659196049531 eV.
<S2> value : 2.0000019578861177.
The main and second norm:
0.9998726757879338 0.00012355990577319796
The main and second norm to orbital pair:
(1.0, 9.0) (1.0, 5.0)


None
Excited energy : 4.665056167805596 eV.
<S2> value : 1.9999525061539698.
The main and second norm:
0.9999994310191987 4.969863399811645e-07
The main and

In [5]:
from pyscf import gto, dft, sftda
import sys
sys.path.append('/home/peter/Research/ZeroExcitationEnergy/code/TwoGGAs')
from GGAnew import eval_xc, td, verify

atom = '''
6  0.000000    0.000000    0.103187
1 -0.000000    0.995526   -0.309561
1 -0.000000   -0.995526   -0.309561
'''
mol = gto.M(atom=atom, charge=0, spin=2, basis='6-31G', verbose=9, symmetry=True)
mf = dft.UKS(mol)
# mf = mf.define_xc_(eval_xc, 'GGA')
mf.xc = 'Slater,'
mf.grids.level = 1
mf.conv_tol = 1e-12
mf.conv_tol_grad = 1e-8
mf.kernel()

System: uname_result(system='Linux', node='peter7920', release='5.15.0-136-generic', version='#147~20.04.1-Ubuntu SMP Wed Mar 19 16:13:14 UTC 2025', machine='x86_64')  Threads 40
Python 3.9.21 (main, Dec 11 2024, 16:24:11) 
[GCC 11.2.0]
numpy 1.26.4  scipy 1.13.1  h5py 3.13.0
Date: Sat May  3 09:47:06 2025
PySCF version 2.8.0
PySCF path  /home/peter/anaconda3/envs/sftdgrad/lib/python3.9/site-packages/pyscf

[CONFIG] ARGPARSE = False
[CONFIG] DEBUG = False
[CONFIG] MAX_MEMORY = 4000
[CONFIG] TMPDIR = /tmp
[CONFIG] UNIT = angstrom
[CONFIG] VERBOSE = 3
[CONFIG] conf_file = None
[INPUT] verbose = 9
[INPUT] max_memory = 4000 
[INPUT] num. atoms = 3
[INPUT] num. electrons = 8
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 2
[INPUT] symmetry True subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 C      0.000000000000   0.000000000000   0.10318

symmetry axis y: [0. 1. 0.]
symmetry axis z: [0. 0. 1.]
num. orbitals of irrep A1 = 7
num. orbitals of irrep B1 = 2
num. orbitals of irrep B2 = 4
number of shells = 9
number of NR pGTOs = 30
number of NR cGTOs = 13
basis = 6-31G
ecp = {}
bas 0, expnt(s) = [3047.5249    457.36951   103.94869    29.210155    9.286663    3.163927]
bas 1, expnt(s) = [7.8682724 1.8812885 0.5442493]
bas 2, expnt(s) = [0.1687144]
bas 3, expnt(s) = [7.8682724 1.8812885 0.5442493]
bas 4, expnt(s) = [0.1687144]
bas 5, expnt(s) = [18.731137   2.8253937  0.6401217]
bas 6, expnt(s) = [0.1612778]
bas 7, expnt(s) = [18.731137   2.8253937  0.6401217]
bas 8, expnt(s) = [0.1612778]
CPU time:       127.18
arg.atm = [[ 6 20  1 23  0  0]
 [ 1 24  1 27  0  0]
 [ 1 28  1 31  0  0]]
arg.bas = [[ 0  0  6  1  0 32 38  0]
 [ 0  0  3  1  0 44 47  0]
 [ 0  0  1  1  0 50 51  0]
 [ 0  1  3  1  0 52 55  0]
 [ 0  1  1  1  0 58 59  0]
 [ 1  0  3  1  0 60 63  0]
 [ 1  0  1  1  0 66 67  0]
 [ 2  0  3  1  0 60 63  0]
 [ 2  0  1  1  0 66 6

-38.27123710390292

In [6]:
from pyscf.sftda.uhf_sf import get_ab_sf

a, b = get_ab_sf(mf)
A_baba, A_abab = a
B_baab, B_abba = b

n_occ_a, n_virt_b = A_abab.shape[0], A_abab.shape[1]
n_occ_b, n_virt_a = B_abba.shape[2], B_abba.shape[3]

A_abab_2d = A_abab.reshape((n_occ_a*n_virt_b,n_occ_a*n_virt_b), order='C')
B_abba_2d = B_abba.reshape((n_occ_a*n_virt_b,n_occ_b*n_virt_a), order='C')
B_baab_2d = B_baab.reshape((n_occ_b*n_virt_a,n_occ_a*n_virt_b), order='C')
A_baba_2d = A_baba.reshape((n_occ_b*n_virt_a,n_occ_b*n_virt_a), order='C')

Casida_matrix = np.block([[ A_abab_2d, B_abba_2d],
                        [-B_baab_2d,-A_baba_2d]])

eigenvals, eigenvecs = np.linalg.eig(Casida_matrix)

idxt = eigenvals.real.argsort()
eigenvals = eigenvals[idxt]
eigenvecs = eigenvecs[:, idxt]

norms = np.linalg.norm(eigenvecs[:n_occ_a * n_virt_b], axis=0)**2
norms -= np.linalg.norm(eigenvecs[n_occ_a * n_virt_b:], axis=0)**2

sfd_eigenvals = eigenvals[norms>0]
sfd_eigenvecs = eigenvecs[:, norms>0].transpose(1,0)

def norm_xy(z):
    x = z[:n_occ_a*n_virt_b].reshape(n_occ_a,n_virt_b)
    y = z[n_occ_a*n_virt_b:].reshape(n_occ_b,n_virt_a)
    norm = np.linalg.norm(x)**2 - np.linalg.norm(y)**2
    norm = np.sqrt(1./norm)
    return ((0,x*norm), (y*norm,0))

mftd1 = sftda.TDA_SF(mf)
mftd1.nstates = len(sfd_eigenvals)
mftd1.extype = 1
mftd1.collinear_samples = 200
mftd1.e = sfd_eigenvals
mftd1.xy = [norm_xy(z) for z in sfd_eigenvecs]

print(transition_analyze(mf, mftd1, mftd1.e[0], mftd1.xy[0], tdtype='TDDFT'))
print(transition_analyze(mf, mftd1, mftd1.e[1], mftd1.xy[1], tdtype='TDDFT'))
print(transition_analyze(mf, mftd1, mftd1.e[2], mftd1.xy[2], tdtype='TDDFT'))
print(transition_analyze(mf, mftd1, mftd1.e[3], mftd1.xy[3], tdtype='TDDFT'))
print(transition_analyze(mf, mftd1, mftd1.e[4], mftd1.xy[4], tdtype='TDDFT'))

200
Excited energy : -8.159843278861037e-09 eV.
<S2> value : 2.051528962978601.
The main and second norm:
0.49752187752819826 0.4902333680357691
The main and second norm to orbital pair:
(5.0, 2.0) (4.0, 1.0)


None
Excited energy : 1.9550719218762014 eV.
<S2> value : 0.00649701036634065.
The main and second norm:
0.7844746604144993 0.21196647020274872
The main and second norm to orbital pair:
(5.0, 1.0) (4.0, 2.0)


None
Excited energy : 2.2301822457472262 eV.
<S2> value : 0.007560395066620096.
The main and second norm:
0.498808479688806 0.4985529961416714
The main and second norm to orbital pair:
(4.0, 1.0) (5.0, 2.0)


None
Excited energy : 4.418170308204107 eV.
<S2> value : 0.018183681826223985.
The main and second norm:
0.7867170416497381 0.21293845120358745
The main and second norm to orbital pair:
(4.0, 2.0) (5.0, 1.0)


None
Excited energy : 7.675620676323738 eV.
<S2> value : 0.9972293069436171.
The main and second norm:
0.9869158723437791 0.011714639405639896
The main and seco